In [31]:
import tree_sitter_python as tspython
from tree_sitter import Language, Parser
from typing import Optional
import traceback

In [32]:
def get_ast_height(python_code: str) -> int:
    """
    Calculate the height of the AST for a Python method using tree-sitter.
    
    Args:
        python_code (str): Python method code as a string
        
    Returns:
        int: Height of the AST (maximum depth from root to any leaf)
    """
    # Initialize the parser with Python language
    PY_LANGUAGE = Language(tspython.language())
    parser = Parser(PY_LANGUAGE)
    
    # Parse the code
    tree = parser.parse(bytes(python_code, "utf8"))
    
    def calculate_height(node):
        """Recursively calculate the height of a tree-sitter node."""
        if node.child_count == 0:
            return 1
        
        max_child_height = 0
        for child in node.children:
            child_height = calculate_height(child)
            max_child_height = max(max_child_height, child_height)
        
        return max_child_height + 1
    
    return calculate_height(tree.root_node)


def analyze_code_with_details(python_code: str) -> dict:
    """
    Analyze code and return detailed information including AST height.
    """
    try:
        height = get_ast_height(python_code)
        
        PY_LANGUAGE = Language(tspython.language())
        parser = Parser(PY_LANGUAGE)
        tree = parser.parse(bytes(python_code, "utf8"))
        
        # Count different node types
        node_counts = {}
        def count_nodes(node):
            node_type = node.type
            node_counts[node_type] = node_counts.get(node_type, 0) + 1
            for child in node.children:
                count_nodes(child)
        
        count_nodes(tree.root_node)
        
        return {
            'height': height,
            'total_nodes': sum(node_counts.values()),
            'node_types': len(node_counts),
            'most_common_nodes': sorted(node_counts.items(), key=lambda x: x[1], reverse=True)[:5],
            'has_errors': tree.root_node.has_error,
            'lines_of_code': len(python_code.strip().split('\n'))
        }
    except Exception as e:
        return {'error': str(e), 'traceback': traceback.format_exc()}

In [33]:
sample_code = """
def fibonacci(n):
    if n <= 1:
        return n
    else:
        return fibonacci(n-1) + fibonacci(n-2)
"""

print("=== Quick Test ===")
height = get_ast_height(sample_code)
print(f"Fibonacci function AST height: {height}")

# Detailed analysis
analysis = analyze_code_with_details(sample_code)
print(f"\nDetailed analysis:")
for key, value in analysis.items():
    if key != 'most_common_nodes':
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value[:5]}...")  # Show top 3

=== Quick Test ===
Fibonacci function AST height: 12

Detailed analysis:
  height: 12
  total_nodes: 47
  node_types: 23
  most_common_nodes: [('identifier', 8), ('(', 3), (')', 3), (':', 3), ('block', 3)]...
  has_errors: False
  lines_of_code: 5


In [34]:
def test_code_height(description: str, code: str, expected_min_height: Optional[int] = None):
    """
    Test function with output formatting.
    """
    print(f"\n{'='*50}")
    print(f"TEST: {description}")
    print(f"{'='*50}")
    
    print("Code:")
    print("-" * 20)
    print(code)
    print("-" * 20)
    
    try:
        analysis = analyze_code_with_details(code)
        
        if 'error' in analysis:
            print(f"❌ Error: {analysis['error']}")
            return False
        
        height = analysis['height']
        print(f"📏 AST Height: {height}")
        print(f"📊 Total Nodes: {analysis['total_nodes']}")
        print(f"🔧 Node Types: {analysis['node_types']}")
        print(f"📝 Lines of Code: {analysis['lines_of_code']}")
        print(f"⚠️  Has Syntax Errors: {analysis['has_errors']}")
        
        if expected_min_height and height < expected_min_height:
            print(f"⚠️  Warning: Height {height} is less than expected minimum {expected_min_height}")
        
        print("Top node types:", [f"{name}({count})" for name, count in analysis['most_common_nodes'][:5]])
        
        return True
        
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        return False

In [35]:
print("Running Simple Test Cases...")

# Test 1: Simple function
test_code_height(
    "Simple Function",
    """
def hello():
    return "world"
""",
    expected_min_height=3
)

# Test 2: Empty function
test_code_height(
    "Empty Function",
    """
def empty():
    pass
""",
    expected_min_height=3
)

Running Simple Test Cases...

TEST: Simple Function
Code:
--------------------

def hello():
    return "world"

--------------------
📏 AST Height: 6
📊 Total Nodes: 15
🔧 Node Types: 15
📝 Lines of Code: 2
⚠️  Has Syntax Errors: False
Top node types: ['module(1)', 'function_definition(1)', 'def(1)', 'identifier(1)', 'parameters(1)']

TEST: Empty Function
Code:
--------------------

def empty():
    pass

--------------------
📏 AST Height: 5
📊 Total Nodes: 11
🔧 Node Types: 11
📝 Lines of Code: 2
⚠️  Has Syntax Errors: False
Top node types: ['module(1)', 'function_definition(1)', 'def(1)', 'identifier(1)', 'parameters(1)']


True

In [36]:
print("Running Complex Test Cases...")

# Test 3: Nested structures
test_code_height(
    "Nested Control Structures",
    """
def complex_function(x, y):
    if x > 0:
        for i in range(10):
            if i % 2 == 0:
                try:
                    result = x + y * i
                    return result
                except Exception as e:
                    print(f"Error: {e}")
    else:
        return None
""",
    expected_min_height=8
)

# Test 4: Class with methods
test_code_height(
    "Class with Methods",
    """
class Calculator:
    def __init__(self, value=0):
        self.value = value
    
    def add(self, x):
        self.value += x
        return self.value
"""
)

Running Complex Test Cases...

TEST: Nested Control Structures
Code:
--------------------

def complex_function(x, y):
    if x > 0:
        for i in range(10):
            if i % 2 == 0:
                try:
                    result = x + y * i
                    return result
                except Exception as e:
                    print(f"Error: {e}")
    else:
        return None

--------------------
📏 AST Height: 18
📊 Total Nodes: 91
🔧 Node Types: 47
📝 Lines of Code: 11
⚠️  Has Syntax Errors: False
Top node types: ['identifier(16)', ':(7)', 'block(7)', 'integer(4)', '((3)']

TEST: Class with Methods
Code:
--------------------

class Calculator:
    def __init__(self, value=0):
        self.value = value

    def add(self, x):
        self.value += x
        return self.value

--------------------
📏 AST Height: 9
📊 Total Nodes: 53
🔧 Node Types: 23
📝 Lines of Code: 7
⚠️  Has Syntax Errors: False
Top node types: ['identifier(15)', ':(3)', 'block(3)', 'attribute(3)', '.(3)']


True

In [37]:
print("Testing Fault Tolerance...")

# Test 5: Syntax errors
test_code_height(
    "Syntax Error Tolerance",
    """
def broken_function(x:
    if x > 0
        return x * 2
    else:
        return 0
    # Missing closing parenthesis and colon
"""
)

# Test 6: Incomplete code
test_code_height(
    "Incomplete Code",
    """
def incomplete(x, y
    if x:
        result = x +
"""
)

Testing Fault Tolerance...

TEST: Syntax Error Tolerance
Code:
--------------------

def broken_function(x:
    if x > 0
        return x * 2
    else:
        return 0
    # Missing closing parenthesis and colon

--------------------
📏 AST Height: 10
📊 Total Nodes: 30
🔧 Node Types: 15
📝 Lines of Code: 6
⚠️  Has Syntax Errors: True
Top node types: ['identifier(8)', 'ERROR(4)', 'type(3)', 'integer(3)', ':(2)']

TEST: Incomplete Code
Code:
--------------------

def incomplete(x, y
    if x:
        result = x +

--------------------
📏 AST Height: 4
📊 Total Nodes: 17
🔧 Node Types: 10
📝 Lines of Code: 3
⚠️  Has Syntax Errors: True
Top node types: ['identifier(7)', 'ERROR(2)', 'module(1)', 'def(1)', '((1)']


True